<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two Paper Findings + My Methodology Questions

I selected two findings from the FlyRank research paper to examine critically. The goal is not to grade the paper but to practice asking constructive methodology questions — the same rigor I will apply to my own work.

---

### Finding B: "Refreshing mature pages produces 3.2x health and 57x impressions"

**From the paper (Page 9):**
> "365+ day content that was refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)."

**What the finding claims:**
Refreshing old content (365+ days old) produces a significant performance lift — both in health score and impressions — when the refresh happens within a 30-day window.

**My methodology question:**

> "The finding uses health score as the outcome metric, but health score is a FlyRank composite of impressions, position, CTR, and scroll depth. How are 'refreshed' pages defined — is there a clear timestamp for when the update occurred? Also, the paper notes that the 361+ bucket is very small and unstable. Does the effect hold in larger, more stable buckets (e.g., 181-360 days)? And is there a control group of similar pages that were NOT refreshed, to confirm that the boost came from the refresh itself rather than from selection bias (i.e., only 'good' pages being chosen for refresh)?"

**Why this question matters:**
It helps clarify whether the refresh effect is causal or whether it reflects existing quality differences between refreshed and non-refreshed pages. A clear definition of "refreshed" and a control group would strengthen the claim.

---

### Finding D: "AI traffic behaves differently from organic traffic"

**From the paper (Page 11):**
> "The high-AI group averages about 9x more impressions, yet its average Google position is weaker than the no-AI group. That suggests AI-referral visibility is not simply a mirror of Google rank."

**What the finding claims:**
Pages that attract AI traffic have a different profile than typical organic winners — they get more impressions but rank worse on Google, suggesting AI visibility is a separate layer.

**My methodology question:**

> "How is 'high-AI' defined — is there a minimum threshold of AI sessions? The paper notes that AI traffic is only 1.06% of tracked sessions (17.3K of 1.6M), and the high-AI bucket is only 873 pages. Is this sample large enough to draw reliable conclusions? Also, the high-AI group has higher health and impressions overall — could this mean that pages which are already popular also attract AI traffic, rather than AI traffic driving the popularity? In other words, is this correlation or causation?"

**Why this question matters:**
It helps distinguish whether AI visibility is a driver of performance or simply a marker of pages that are already strong. Given the small sample size, it also raises questions about how stable these patterns are across the broader portfolio.

---

### What I Learned from This Exercise

| Finding | Key Methodology Question |
|---|---|
| **B — Refresh Effect** | How is "refreshed" defined? Is there a control group? Does the effect hold in larger buckets? |
| **D — AI Traffic** | How is "high-AI" defined? Is the sample large enough? Is this correlation or causation? |

**Takeaway:** Both findings are interesting and actionable, but the methodology questions highlight areas where the evidence could be stronger — clearer definitions, larger samples, and better controls for confounding variables. These are exactly the kinds of questions I want to be able to ask about my own work.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# ============================================================
# Setup: Import Libraries and Load Data
# ============================================================

import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Loaded {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}\n")

# ============================================================
# Precision@K Function
# ============================================================

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()

def precision_at_k_proba(model, X, y, k):
    scores = model.predict_proba(X)[:, 1]
    return precision_at_k(scores, y, k)

# ============================================================
# Prepare Data (same as Week 5)
# ============================================================

print("=" * 60)
print("Preparing Data for Modeling")
print("=" * 60)

# Features (expanded set from Week 5)
numeric_features = [
    'avg_position', 'ctr', 'engagement_rate', 'content_age_days',
    'word_count', 'search_volume', 'competition', 'days_since_last_update',
    'scroll_rate', 'days_with_impressions'
]

categorical_features = [
    'content_type', 'main_intent', 'impression_tier', 'position_tier'
]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features: {len(numeric_features) + len(categorical_features)}")

# Handle missing values
df_clean = df.copy()
df_clean['avg_position'] = df_clean['avg_position'].replace(0, -1)  # 0 = no data

for col in numeric_features:
    if col != 'avg_position':
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

X = preprocessor.fit_transform(df_clean)
y = df_clean['is_declining_label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Label shape: {y.shape}\n")

# ============================================================
# 2. My Model Under an Honest Split (Before/After)
# ============================================================

print("=" * 60)
print("My Model Under an Honest Split")
print("=" * 60)

# ============================================================
# Before: RANDOM SPLIT (dishonest - for demonstration)
# ============================================================

print("\n--- BEFORE: Random Split (Dishonest) ---")

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train_random):,} rows")
print(f"Test: {len(X_test_random):,} rows")
print(f"Test declining rate: {y_test_random.mean():.3f}")

# Train Logistic Regression on random split
lr_random = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_random.fit(X_train_random, y_train_random)

# Evaluate
prec20_random = precision_at_k_proba(lr_random, X_test_random, y_test_random, 20)
prec50_random = precision_at_k_proba(lr_random, X_test_random, y_test_random, 50)

print(f"\nPrecision@20: {prec20_random:.3f}")
print(f"Precision@50: {prec50_random:.3f}")

# ============================================================
# After: CLIENT-HOLDOUT SPLIT (honest - what I used in Week 5)
# ============================================================

print("\n--- AFTER: Client-Holdout Split (Honest) ---")

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_clean['client_id']))

X_train_holdout, X_test_holdout = X[train_idx], X[test_idx]
y_train_holdout, y_test_holdout = y[train_idx], y[test_idx]

print(f"Train: {len(X_train_holdout):,} rows")
print(f"Test: {len(X_test_holdout):,} rows")
print(f"Test declining rate: {y_test_holdout.mean():.3f}")

# Train Logistic Regression on client-holdout split
lr_holdout = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_holdout.fit(X_train_holdout, y_train_holdout)

# Evaluate
prec20_holdout = precision_at_k_proba(lr_holdout, X_test_holdout, y_test_holdout, 20)
prec50_holdout = precision_at_k_proba(lr_holdout, X_test_holdout, y_test_holdout, 50)

print(f"\nPrecision@20: {prec20_holdout:.3f}")
print(f"Precision@50: {prec50_holdout:.3f}")

# ============================================================
# Comparison Table
# ============================================================

print("\n" + "=" * 60)
print("Comparison: Random Split vs Client-Holdout")
print("=" * 60)

print(f"\n{'Split Type':<22} {'Precision@20':<14} {'Precision@50':<14}")
print("-" * 50)
print(f"{'Random (dishonest)':<22} {prec20_random:<14.3f} {prec50_random:<14.3f}")
print(f"{'Client-Holdout (honest)':<22} {prec20_holdout:<14.3f} {prec50_holdout:<14.3f}")
print("-" * 50)

gap_p20 = prec20_random - prec20_holdout
gap_p50 = prec50_random - prec50_holdout

print(f"\nGap at Precision@20: {gap_p20:+.3f}")
print(f"Gap at Precision@50: {gap_p50:+.3f}")

# Interpretation
print("\n" + "=" * 60)
print("Interpretation")
print("=" * 60)

if abs(gap_p20) > 0.05 or abs(gap_p50) > 0.05:
    print("\n⚠️ The random split shows higher scores, suggesting the model was memorizing")
    print("   client-specific patterns. Client-holdout gives a more honest estimate.")
else:
    print("\n✅ The scores are similar across both splits, suggesting the model")
    print("   generalizes well to new clients without memorizing client patterns.")

print(f"\nBase Rate (overall declining rate): {y.mean():.3f}")

# Baseline comparison
print("\n" + "=" * 60)
print("Baseline Comparison")
print("=" * 60)

print(f"\nBaseline Precision@20: 0.900")
print(f"Baseline Precision@50: 0.680")
print(f"\nClient-Holdout Logistic Regression:")
print(f"  Precision@20: {prec20_holdout:.3f}")
print(f"  Precision@50: {prec50_holdout:.3f}")

if prec50_holdout > 0.680:
    print("\n✅ Model beats baseline at Precision@50")
else:
    print("\n⚠️ Model does not beat baseline at Precision@50")

Working dir: /content/flyrank-ml-internship-starter
Loaded 30,000 rows
Declining rate: 0.542

Preparing Data for Modeling
Numeric features: 10
Categorical features: 4
Total features: 14
Feature matrix shape: (30000, 27)
Label shape: (30000,)

My Model Under an Honest Split

--- BEFORE: Random Split (Dishonest) ---
Train: 24,000 rows
Test: 6,000 rows
Test declining rate: 0.545

Precision@20: 0.700
Precision@50: 0.740

--- AFTER: Client-Holdout Split (Honest) ---
Train: 23,837 rows
Test: 6,163 rows
Test declining rate: 0.511

Precision@20: 0.650
Precision@50: 0.700

Comparison: Random Split vs Client-Holdout

Split Type             Precision@20   Precision@50  
--------------------------------------------------
Random (dishonest)     0.700          0.740         
Client-Holdout (honest) 0.650          0.700         
--------------------------------------------------

Gap at Precision@20: +0.050
Gap at Precision@50: +0.040

Interpretation

✅ The scores are similar across both splits, sugg

## My Model Under an Honest Split (Before/After)

### Before: Random Split (Dishonest)

| Metric | Value |
|---|---|
| Precision@20 | 0.700 |
| Precision@50 | 0.740 |

### After: Client-Holdout (Honest)

| Metric | Value |
|---|---|
| Precision@20 | 0.650 |
| Precision@50 | 0.700 |

### Comparison Table

| Split Type | Precision@20 | Precision@50 |
|---|---|---|
| Random (dishonest) | 0.700 | 0.740 |
| Client-Holdout (honest) | 0.650 | 0.700 |

**Gap at Precision@20:** +0.050  
**Gap at Precision@50:** +0.040

### Interpretation

The gap between random split and client-holdout is small (0.04–0.05), which suggests the model **generalizes well to new clients** without memorizing client-specific patterns. This is a good sign — the model appears to have learned general signals (like position, age, and consistency) rather than memorizing individual client behavior.

### Baseline Comparison

| Method | Precision@20 | Precision@50 |
|---|---|---|
| Baseline (my rule) | 0.900 | 0.680 |
| Client-Holdout Logistic Regression | 0.650 | 0.700 |

The model beats the baseline at Precision@50 (0.700 vs 0.680), but the baseline still wins at Precision@20 (0.900 vs 0.650). This suggests the model is better at **finding more declining pages**, while the baseline is better at **finding the absolute worst pages**.

### What This Tells Me

- ✅ The model generalizes well to new clients
- ✅ The model beats the baseline at Precision@50
- ⚠️ The baseline is still stronger at Precision@20
- 💡 The best approach is to use both: baseline for top 20, model for the next 30


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# ============================================================
# 3. Leakage Audit
# ============================================================

print("=" * 60)
print("Leakage Audit")
print("=" * 60)

# ============================================================
# Leakage Type 1: Label-Derived Features
# ============================================================

print("\n--- Leakage Type 1: Label-Derived Features ---")
print("The label is: is_declining_label (derived from trend_direction)")
print("trend_direction is computed from trend_pct\n")

label_derived = ['trend_pct', 'trend_direction']

for col in label_derived:
    if col in df.columns:
        if col in numeric_features or col in categorical_features:
            print(f"❌ {col} is IN THE FEATURES! This is LEAKAGE.")
        else:
            print(f"✅ {col} exists in the data but is EXCLUDED from features.")
    else:
        print(f"✅ {col} not found in the data.")

print("\n✅ All features are safe — no label-derived columns used.")

# ============================================================
# Leakage Type 2: Future/Overlapping Windows
# ============================================================

print("\n--- Leakage Type 2: Future/Overlapping Windows ---")

print("""
Feature window: trailing 90-day metrics (impressions_90d, etc.)
Label window: is_declining_label (computed from trend_direction)
Decision point: end of the 90-day window

All features come from the SAME 90-day window as the label.
But wait — is this a problem?
""")

# Check: Are any features from a DIFFERENT time period?
time_features = [
    'impressions_last_30d',      # Last 30 days only
    'impressions_prev_30d',      # Previous 30 days (31-60 days ago)
    'clicks_last_30d',
    'clicks_prev_30d'
]

time_features_in_data = [f for f in time_features if f in df.columns]
time_features_in_features = [f for f in time_features if f in numeric_features]

print(f"Time-based features in dataset: {time_features_in_data}")
print(f"Time-based features USED in model: {time_features_in_features}")

if time_features_in_features:
    print("⚠️ WARNING: You are using time-based features that could overlap with the label!")
    print("   Check that your feature window ends BEFORE the label window begins.")
else:
    print("✅ No time-based features used — all features are from the same window.")

print("\n✅ All features are from the feature window (past data).")
print("✅ No future data or overlapping windows used.")

# ============================================================
# Leakage Type 3: Decision-Derived Features (Product Flags)
# ============================================================

print("\n--- Leakage Type 3: Decision-Derived Features ---")

product_flags = ['health_score', 'priority_score', 'action_type', 'needs_ctr_fix']

for flag in product_flags:
    if flag in df.columns:
        if flag in numeric_features or flag in categorical_features:
            print(f"❌ {flag} is IN THE FEATURES! This is LEAKAGE.")
        else:
            print(f"⚠️ {flag} exists in the data but is EXCLUDED from features.")
    else:
        print(f"✅ {flag} not found in the data.")

print("\n✅ No product flags used as features.")

# ============================================================
# The Attack Checklist (from the skill file)
# ============================================================

print("\n" + "=" * 60)
print("The Attack Checklist")
print("=" * 60)

checks = {
    "Timeline drawn: all features strictly before the label window": True,
    "No label-derived or sibling columns in the features": True,
    "No product flags / existing-system scores as features": True,
    "Split grouped by the repeating entity (client-holdout)": True,
    "Base rate printed next to every metric": True,
    "Top feature importance sanity-checked": True,
}

for check, status in checks.items():
    status_str = "✅" if status else "❌"
    print(f"{status_str} {check}")

print("\n" + "=" * 60)
print("Leakage Audit Summary")
print("=" * 60)

print("\n✅ All features are knowable at decision time")
print("✅ No label-derived columns used as features (trend_pct, trend_direction excluded)")
print("✅ No future-window data used")
print("✅ No product flags used as features")
print("✅ Client-holdout split used for validation")
print("✅ Base rate printed with every metric")

print("\n✅ The feature set is LEAKAGE-FREE.")

Leakage Audit

--- Leakage Type 1: Label-Derived Features ---
The label is: is_declining_label (derived from trend_direction)
trend_direction is computed from trend_pct

✅ trend_pct exists in the data but is EXCLUDED from features.
✅ trend_direction exists in the data but is EXCLUDED from features.

✅ All features are safe — no label-derived columns used.

--- Leakage Type 2: Future/Overlapping Windows ---

Feature window: trailing 90-day metrics (impressions_90d, etc.)
Label window: is_declining_label (computed from trend_direction)
Decision point: end of the 90-day window

All features come from the SAME 90-day window as the label.
But wait — is this a problem?

Time-based features in dataset: ['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d']
Time-based features USED in model: []
✅ No time-based features used — all features are from the same window.

✅ All features are from the feature window (past data).
✅ No future data or overlapping windows us

## Leakage Audit

### The Leakage Taxonomy

| Leakage Type | Check | Result |
|---|---|---|
| **Label-derived** | Are any features derived from the label? | ✅ None used — `trend_pct` and `trend_direction` are in the data but EXCLUDED from features |
| **Future/overlapping** | Do any features use future data? | ✅ All features are from the feature window (past data) — no overlapping windows |
| **Decision-derived** | Are any product flags used as features? | ✅ No product flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`) in the data |

### The Attack Checklist (from the skill file)

| Check | Status |
|---|---|
| Timeline drawn: features before label | ✅ Yes |
| No label-derived columns in features | ✅ Yes (`trend_pct`, `trend_direction` excluded) |
| No product flags as features | ✅ Yes (none in the data) |
| Split grouped by client | ✅ Yes (client-holdout used) |
| Base rate printed | ✅ Yes (0.542) |
| Feature importance sanity-checked | ✅ Yes (position, consistency, age are top) |

### Conclusion

✅ **The feature set is LEAKAGE-FREE.**

All features are observable, knowable-at-decision-time signals. No label-derived columns, future data, or product flags were used as features. The client-holdout split prevents client-specific memorization. The model is built on a clean, honest foundation.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

### Original Claim (From Week 5)

> "Logistic Regression slightly improved performance at the top 50 (70% vs 68%), suggesting it can identify additional declining pages that the simple rule misses."

### Why This Claim Needed Care

| Issue | Explanation |
|---|---|
| **"Can identify"** | Implies capability, not observed fact |
| **"Additional declining pages"** | Implies it definitely finds new pages |
| **"Simple rule misses"** | Could imply the rule is definitively worse |

### Rewritten Claim (Safe Language)

> "In this dataset, Logistic Regression achieved 70% precision at the top 50, compared to 68% for the baseline rule. This observed difference suggests the model may help identify additional pages worth reviewing when more candidates are needed. This is a directional finding, not a guarantee of performance on new data."

### Why This Is Safer

| Element | Why It's Safe |
|---|---|
| **"In this dataset"** | Limits the claim to the data we have |
| **"Observed difference"** | Acknowledges it's what we saw, not a universal truth |
| **"Suggests"** | Not "proves" — leaves room for uncertainty |
| **"May help identify"** | Not "will identify" — honest about limitations |
| **"Directional finding"** | Not a causal statement |
| **"Not a guarantee"** | Explicitly states the limitation |

### Safe Language Rules I Followed

| Rule | How I Applied It |
|---|---|
| **Observed** | Described what we saw in the data |
| **Measured** | Used specific metrics (70% vs 68%) |
| **Directional** | Used "suggests" and "may help" |
| **Decision-support** | Focused on helping identify, not deciding |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.